In [3]:
import pandas as pd

In [4]:
df = pd.read_csv(r"C:\Users\LENOVO\Downloads\ASE-ETL-dataset_1_1 (1)\employee_202510161125.csv",sep="|")

In [5]:
df.shape

(50, 34)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   client_employee_id     50 non-null     int64  
 1   first_name             50 non-null     object 
 2   middle_name            0 non-null      float64
 3   last_name              50 non-null     object 
 4   preferred_name         50 non-null     object 
 5   job_code               50 non-null     int64  
 6   job_title              50 non-null     object 
 7   job_start_date         50 non-null     object 
 8   organization_id        50 non-null     object 
 9   organization_name      50 non-null     object 
 10  department_id          50 non-null     object 
 11  department_name        50 non-null     object 
 12  dob                    50 non-null     object 
 13  hire_date              50 non-null     object 
 14  recent_hire_date       50 non-null     object 
 15  annivers

In [7]:
employee_ids = set(df['client_employee_id'])
invalid_managers = df[
    df['manager_employee_id'].notna() &
    ~df['manager_employee_id'].isin(employee_ids)
]

print(invalid_managers)

    client_employee_id  first_name  middle_name last_name preferred_name  \
0                   37   Demo00037          NaN     Nurse      Demo00037   
1               507210  Demo507210          NaN     Nurse     Demo507210   
2                   80   Demo00080          NaN     Nurse      Demo00080   
3               503912  Demo503912          NaN     Nurse     Demo503912   
5               506323  Demo506323          NaN     Nurse     Demo506323   
6                   68   Demo00068          NaN     Nurse      Demo00068   
8                  131   Demo00131          NaN     Nurse      Demo00131   
9                  214   Demo00214          NaN     Nurse      Demo00214   
10              505453  Demo505453          NaN     Nurse     Demo505453   
12                 138   Demo00138          NaN     Nurse      Demo00138   
13              507273  Demo507273          NaN     Nurse     Demo507273   
15              507238  Demo507238          NaN     Nurse     Demo507238   
16          

In [8]:
df['organization_name'].unique()

array(['Lowell General Hospital', 'Care at Home', 'Tufts Medicine'],
      dtype=object)

In [9]:
# Group by employee ID and count the number of unique organizations (or organization names)
emp_org_counts = df.groupby('client_employee_id')['organization_name'].nunique()

# Get the IDs of employees who have more than 1 unique organization
multi_org_emp_ids = emp_org_counts[emp_org_counts > 1].index

# Filter the original dataframe to show only the records of those employees
multi_org_employees = df[df['client_employee_id'].isin(multi_org_emp_ids)].sort_values('client_employee_id')

# Display the result
multi_org_employees[['client_employee_id', 'first_name', 'last_name', 'organization_name']]

,client_employee_id,first_name,last_name,organization_name


In [10]:
dept_manager_count = df.groupby('department_id')['manager_employee_id'].nunique()

print(dept_manager_count)

department_id
209037        1
209038        1
214401        1
214638        1
217001        1
220001        1
220537        1
227001        1
235540        1
236037        1
236045        1
236082        1
241028        1
241037        2
246037        2
246045        1
247037        2
247045        1
250045        1
60700         1
60900         1
61100         1
63200         1
68250         1
69000         1
69200         1
71200         1
72300         1
72640         1
72800         2
73480         1
73600         1
75000         1
8000-13204    1
8000-13206    1
8000-14678    1
90420         1
98230         1
Name: manager_employee_id, dtype: int64


In [11]:
# Check Employee <-> Organization
emp_multi_org = (df.groupby('client_employee_id')['organization_id'].nunique() > 1).any()
org_multi_emp = (df.groupby('organization_id')['client_employee_id'].nunique() > 1).any()
print(f"Employee <-> Organization Many-to-Many: {emp_multi_org and org_multi_emp}")

# Check Employee <-> Department
emp_multi_dept = (df.groupby('client_employee_id')['department_id'].nunique() > 1).any()
dept_multi_emp = (df.groupby('department_id')['client_employee_id'].nunique() > 1).any()
print(f"Employee <-> Department Many-to-Many: {emp_multi_dept and dept_multi_emp}")

# Check Employee <-> Manager
emp_multi_mgr = (df.groupby('client_employee_id')['manager_employee_id'].nunique() > 1).any()
mgr_multi_emp = (df.groupby('manager_employee_id')['client_employee_id'].nunique() > 1).any()
print(f"Employee <-> Manager Many-to-Many: {emp_multi_mgr and mgr_multi_emp}")

Employee <-> Organization Many-to-Many: False
Employee <-> Department Many-to-Many: False
Employee <-> Manager Many-to-Many: False


In [12]:
def check_relationship(df, entity1_col, entity2_col, entity1_name, entity2_name):
    # Calculate the maximum number of unique entity2 per entity1
    max_e2_per_e1 = df.groupby(entity1_col)[entity2_col].nunique().max()
    # Calculate the maximum number of unique entity1 per entity2
    max_e1_per_e2 = df.groupby(entity2_col)[entity1_col].nunique().max()
    
    print(f"--- {entity1_name} vs {entity2_name} ---")
    print(f"Max {entity2_name}s per {entity1_name}: {max_e2_per_e1}")
    print(f"Max {entity1_name}s per {entity2_name}: {max_e1_per_e2}")
    
    if max_e2_per_e1 > 1 and max_e1_per_e2 > 1:
        print(f"Relationship: Many-to-Many\n")
    elif max_e2_per_e1 == 1 and max_e1_per_e2 > 1:
        print(f"Relationship: One-to-Many (1 {entity2_name} has Many {entity1_name}s)\n")
    elif max_e2_per_e1 > 1 and max_e1_per_e2 == 1:
        print(f"Relationship: One-to-Many (1 {entity1_name} has Many {entity2_name}s)\n")
    elif max_e2_per_e1 == 1 and max_e1_per_e2 == 1:
        print(f"Relationship: One-to-One\n")
    else:
        print("Dataset might be empty or missing data\n")

check_relationship(df, 'client_employee_id', 'organization_id', 'Employee', 'Organization')
check_relationship(df, 'client_employee_id', 'department_id', 'Employee', 'Department')
check_relationship(df, 'client_employee_id', 'manager_employee_id', 'Employee', 'Manager')

--- Employee vs Organization ---
Max Organizations per Employee: 1
Max Employees per Organization: 30
Relationship: One-to-Many (1 Organization has Many Employees)

--- Employee vs Department ---
Max Departments per Employee: 1
Max Employees per Department: 5
Relationship: One-to-Many (1 Department has Many Employees)

--- Employee vs Manager ---
Max Managers per Employee: 1
Max Employees per Manager: 10
Relationship: One-to-Many (1 Manager has Many Employees)

